<center>
<h1>Лабораторна робота №2</h1>
<h2>Наука про дані: підготовчий етап</h2>
<h3>Частина 2</h3>
</center>

### Завдання 1
Звантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset. Здійснити data cleaning

In [1]:
import pandas as pd
import numpy as np
import timeit
import datetime

df = pd.read_csv('household_power_consumption.txt', sep=';', 
                 na_values='?', low_memory=False)

df = df.dropna()

numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col])

df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Завдання 2 
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт. Виміряти час виконання операції за допомогою модуля timeit.

In [2]:
def filter_power(data):
    return data[data['Global_active_power'] > 5]

start_time = timeit.default_timer()
res_2 = filter_power(df)
time_2 = timeit.default_timer() - start_time

print(f"Кількість знайдених рядків: {len(res_2)}")
print(f"Час виконання: {time_2:.6f} сек")
res_2.head()

Кількість знайдених рядків: 17547
Час виконання: 0.017771 сек


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


### Завдання 3
Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильник (Sub_metering_2) споживають більше, ніж бойлер та кондиціонер (Sub_metering_3).

In [5]:
def filter_intensity_and_appliances(data):
    return data[(data['Global_intensity'] >= 19) & 
                (data['Global_intensity'] <= 20) & 
                (data['Sub_metering_2'] > data['Sub_metering_3'])]

start_time = timeit.default_timer()
res_3 = filter_intensity_and_appliances(df)
time_3 = timeit.default_timer() - start_time

print(f"Кількість знайдених рядків: {len(res_3)}")
print(f"Час виконання: {time_3:.6f} сек")
res_3.head()

Кількість знайдених рядків: 2509
Час виконання: 0.022688 сек


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


### Завдання 4
Обрати випадковим чином 500 000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [6]:
sample_df = df.sample(n=500000, replace=False)
avg_values = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

print("Середні значення споживання по групах:")
print(avg_values)

Середні значення споживання по групах:
Sub_metering_1    1.120066
Sub_metering_2    1.288880
Sub_metering_3    6.466046
dtype: float64


### Завдання 5
Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [8]:
df['Time_obj'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time

evening_df = df[(df['Time_obj'] > datetime.time(18, 0, 0)) & 
                (df['Global_active_power'] > 6) & 
                (df['Sub_metering_2'] > df['Sub_metering_1']) & 
                (df['Sub_metering_2'] > df['Sub_metering_3'])]

mid = len(evening_df) // 2
part_1 = evening_df.iloc[:mid:3]
part_2 = evening_df.iloc[mid::4]

final_selection = pd.concat([part_1, part_2])

print(f"Знайдено записів за умовами: {len(evening_df)}")
print(f"Кількість після вибору кожного 3-го та 4-го:{len(final_selection)}")
final_selection.head()

Знайдено записів за умовами: 1059
Кількість після вибору кожного 3-го та 4-го:310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Time_obj
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,18:05:00
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,18:08:00
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,20:58:00
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,21:02:00
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,21:05:00


### Завдання 6
Пронормувати та стандартувати вибраний датасет. 
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів. 
Провести One Hot Encoding категоріального атрибута.

In [11]:
pearson_val = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman_val = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Кореляція Пірсона: {pearson_val:.4f}")
print(f"Кореляція Спірмена: {spearman_val:.4f}")

p_min, p_max = df['Global_active_power'].min(), df['Global_active_power'].max()
df['Normalized_Power'] = (df['Global_active_power'] - p_min) / (p_max - p_min)
df['Standardized_Power'] = (df['Global_active_power'] - df['Global_active_power'].mean()) / df['Global_active_power'].std()
df['Power_Level'] = np.where(df['Global_active_power'] > 5, 'High', 'Normal')
ohe_result = pd.get_dummies(df['Power_Level'], prefix='Level')

print("\nРезультат One Hot Encoding (перші 5 рядків):")
print(ohe_result.head())

Кореляція Пірсона: 0.9989
Кореляція Спірмена: 0.9954

Результат One Hot Encoding (перші 5 рядків):
   Level_High  Level_Normal
0       False          True
1        True         False
2        True         False
3        True         False
4       False          True
